# WESAD Model Building - Scenario 1 (All Subjects)
This notebook loads the preprocessed data from all subjects, combines them, and trains four machine learning models: Random Forest, SVM, XGBoost, and KNN.
The trained models are then saved to the `Models/Scenario_1-All` directory.

## 1. Import Libraries & Load Preprocessing Functions
We use `%run` to execute the preprocessing notebook, which makes all its functions available in this notebook.

In [2]:
import pickle
import numpy as np
import pandas as pd
import os
import warnings
from scipy.signal import butter, filtfilt
from scipy import stats
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ----------------------------------------------------
# Preprocessing Pipeline Functions (from preprocess.ipynb)
# ----------------------------------------------------

def load_wesad_subject(subject_id, base_path='WESAD'):
    """Loads a single subject's data from the WESAD dataset."""
    file_path = os.path.join(base_path, f'S{subject_id}', f'S{subject_id}.pkl')
    with open(file_path, 'rb') as file:
        data = pickle.load(file, encoding='latin1')
    return data

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def butter_lowpass(cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a

def apply_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    return filtfilt(b, a, data)

def apply_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    return filtfilt(b, a, data)

def filter_signals(data):
    """Applies appropriate filters to all signals in the dataset."""
    filtered_data = {'chest': {}, 'wrist': {}}
    
    # Chest signals (fs = 700 Hz)
    fs_chest = 700
    if 'ECG' in data['signal']['chest']:
        filtered_data['chest']['ECG'] = apply_bandpass_filter(data['signal']['chest']['ECG'].flatten(), 0.5, 50, fs_chest)
    if 'EDA' in data['signal']['chest']:
        filtered_data['chest']['EDA'] = apply_lowpass_filter(data['signal']['chest']['EDA'].flatten(), 5, fs_chest)
    if 'EMG' in data['signal']['chest']:
        filtered_data['chest']['EMG'] = apply_bandpass_filter(data['signal']['chest']['EMG'].flatten(), 10, 250, fs_chest)
    if 'Resp' in data['signal']['chest']:
        filtered_data['chest']['Resp'] = apply_lowpass_filter(data['signal']['chest']['Resp'].flatten(), 0.5, fs_chest, order=3)
        
    # Wrist signals
    if 'BVP' in data['signal']['wrist']:
        fs_bvp = 64
        filtered_data['wrist']['BVP'] = apply_bandpass_filter(data['signal']['wrist']['BVP'].flatten(), 0.5, 8, fs_bvp)
    if 'EDA' in data['signal']['wrist']:
        fs_eda_wrist = 4
        filtered_data['wrist']['EDA'] = apply_lowpass_filter(data['signal']['wrist']['EDA'].flatten(), 1, fs_eda_wrist)
        
    return filtered_data

def create_sliding_windows(signal, fs, window_sec=60, step_sec=30):
    """Segments a signal into fixed sliding windows."""
    window_samples = int(window_sec * fs)
    step_samples = int(step_sec * fs)
    
    windows = []
    for start_idx in range(0, len(signal) - window_samples + 1, step_samples):
        end_idx = start_idx + window_samples
        windows.append(signal[start_idx:end_idx])
        
    return np.array(windows)

def get_window_labels(labels, fs=700, window_sec=60, step_sec=30):
    """Gets the most frequent label for each window."""
    window_samples = int(window_sec * fs)
    step_samples = int(step_sec * fs)
    
    window_labels = []
    for start_idx in range(0, len(labels) - window_samples + 1, step_samples):
        end_idx = start_idx + window_samples
        window_data = labels[start_idx:end_idx]
        mode_result = stats.mode(window_data, keepdims=True)
        window_labels.append(mode_result.mode[0])
        
    return np.array(window_labels)

def segment_all_signals(filtered_data, labels, window_sec=60, step_sec=30):
    """Segments all filtered signals and aligns labels."""
    segmented_data = {'chest': {}, 'wrist': {}}
    
    fs_chest = 700
    for sig_name, sig_data in filtered_data['chest'].items():
        segmented_data['chest'][sig_name] = create_sliding_windows(sig_data, fs_chest, window_sec, step_sec)
        
    if 'BVP' in filtered_data['wrist']:
        segmented_data['wrist']['BVP'] = create_sliding_windows(filtered_data['wrist']['BVP'], 64, window_sec, step_sec)
    if 'EDA' in filtered_data['wrist']:
        segmented_data['wrist']['EDA'] = create_sliding_windows(filtered_data['wrist']['EDA'], 4, window_sec, step_sec)
        
    windowed_labels = get_window_labels(labels, fs_chest, window_sec, step_sec)
    
    return segmented_data, windowed_labels

def extract_statistical_features(window):
    """Extracts basic statistical features from a single window of data."""
    return [
        np.mean(window),
        np.std(window),
        np.min(window),
        np.max(window)
    ]

def extract_features_from_all_signals(segmented_data):
    """Extracts features from all segmented signals and concatenates them."""
    num_windows = len(list(segmented_data['chest'].values())[0])
    all_features = []
    
    for i in range(num_windows):
        window_features = []
        for sig_name, sig_windows in segmented_data['chest'].items():
            window_features.extend(extract_statistical_features(sig_windows[i]))
        for sig_name, sig_windows in segmented_data['wrist'].items():
            window_features.extend(extract_statistical_features(sig_windows[i]))
        all_features.append(window_features)
        
    return np.array(all_features)

def normalize_features(features):
    """Applies Z-score normalization to the feature matrix."""
    scaler = StandardScaler()
    return scaler.fit_transform(features)

def process_subject(subject_id):
    print(f"Processing Subject {subject_id}...")
    data = load_wesad_subject(subject_id)
    labels = data['label']
    filtered_data = filter_signals(data)
    segmented_data, windowed_labels = segment_all_signals(filtered_data, labels)
    features = extract_features_from_all_signals(segmented_data)
    
    valid_mask = np.isin(windowed_labels, [1, 2, 3, 4])
    valid_features = features[valid_mask]
    valid_labels = windowed_labels[valid_mask]
    
    normalized_features = normalize_features(valid_features)
    print(f"  Subject {subject_id} processed: {normalized_features.shape[0]} windows, {normalized_features.shape[1]} features.")
    return normalized_features, valid_labels

print("Pipeline functions defined successfully.")

Pipeline functions defined successfully.


## 2. Process & Combine Data from All Subjects
We iterate through all available subject folders (`S2` to `S17`, excluding missing ones like `S12` if applicable), process each subject using `process_subject()`, and concatenate all features and labels.

In [3]:
# List of subject IDs to process
# WESAD dataset contains subjects S2 to S17 (excluding S12)
subject_ids = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

all_features = []
all_labels = []

for s_id in subject_ids:
    try:
        features, labels = process_subject(s_id)
        all_features.append(features)
        all_labels.append(labels)
    except Exception as e:
        print(f"Skipping Subject {s_id} due to error: {e}")

# Concatenate all features and labels
X = np.vstack(all_features)
y = np.concatenate(all_labels)

print("\n--- Aggregated Dataset Summary ---")
print(f"Total samples (X shape): {X.shape}")
print(f"Total labels (y shape): {y.shape}")
print("Label distribution:")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    label_name = {1: 'Baseline', 2: 'Stress', 3: 'Amusement', 4: 'Meditation'}.get(u, f'Unknown ({u})')
    print(f"  Class {u} ({label_name}): {c} samples")

Processing Subject 2...
  Subject 2 processed: 96 windows, 24 features.
Processing Subject 3...
  Subject 3 processed: 99 windows, 24 features.
Processing Subject 4...
  Subject 4 processed: 99 windows, 24 features.
Processing Subject 5...
  Subject 5 processed: 99 windows, 24 features.
Processing Subject 6...
  Subject 6 processed: 99 windows, 24 features.
Processing Subject 7...
  Subject 7 processed: 99 windows, 24 features.
Processing Subject 8...
  Subject 8 processed: 100 windows, 24 features.
Processing Subject 9...
  Subject 9 processed: 99 windows, 24 features.
Processing Subject 10...
  Subject 10 processed: 103 windows, 24 features.
Processing Subject 11...
  Subject 11 processed: 102 windows, 24 features.
Processing Subject 13...
  Subject 13 processed: 102 windows, 24 features.
Processing Subject 14...
  Subject 14 processed: 102 windows, 24 features.
Processing Subject 15...
  Subject 15 processed: 101 windows, 24 features.
Processing Subject 16...
  Subject 16 processed:

## 3. Train/Test Split
We split the combined dataset into training (80%) and testing (20%) sets.

In [4]:
from sklearn.model_selection import train_test_split

# Split data: 80% train, 20% test
# stratify=y ensures the class distribution is maintained in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: X={X_train.shape}, y={y_train.shape}")
print(f"Testing set shape: X={X_test.shape}, y={y_test.shape}")

Training set shape: X=(1199, 24), y=(1199,)
Testing set shape: X=(300, 24), y=(300,)


## 4. Model Training & Evaluation
We will train four models:
1. Random Forest
2. Support Vector Machine (SVM)
3. XGBoost
4. K-Nearest Neighbors (KNN)

After training, we evaluate each model on the test set.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

# XGBoost requires labels to start from 0. 
# Our labels are 1, 2, 3, 4. We need to map them to 0, 1, 2, 3 for XGBoost.
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

# Initialize models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

trained_models = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"--- Training {name} ---")
    start_time = time.time()
    
    # Train
    if name == "XGBoost":
        model.fit(X_train, y_train_xgb)
    else:
        model.fit(X_train, y_train)
        
    train_time = time.time() - start_time
    
    # Predict
    if name == "XGBoost":
        y_pred_xgb = model.predict(X_test)
        y_pred = y_pred_xgb + 1 # Map back to 1, 2, 3, 4 for evaluation
    else:
        y_pred = model.predict(X_test)
        
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    print(f"Training Time: {train_time:.2f} seconds")
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Baseline', 'Stress', 'Amusement', 'Meditation']))
    print("\n")
    
    # Store trained model
    trained_models[name] = model

--- Training Random Forest ---
Training Time: 0.37 seconds
Accuracy: 0.9533
Classification Report:
              precision    recall  f1-score   support

    Baseline       0.98      0.97      0.98       118
      Stress       0.99      1.00      0.99        66
   Amusement       0.86      0.86      0.86        37
  Meditation       0.92      0.92      0.92        79

    accuracy                           0.95       300
   macro avg       0.94      0.94      0.94       300
weighted avg       0.95      0.95      0.95       300



--- Training SVM ---
Training Time: 0.17 seconds
Accuracy: 0.9000
Classification Report:
              precision    recall  f1-score   support

    Baseline       0.91      0.97      0.94       118
      Stress       0.98      0.98      0.98        66
   Amusement       0.75      0.49      0.59        37
  Meditation       0.86      0.91      0.88        79

    accuracy                           0.90       300
   macro avg       0.88      0.84      0.85      

## 5. Save Models
Finally, we save the trained models to the `Models/Scenario_1-All` directory using `pickle`.

In [6]:
import os
import pickle

# Define save directory
# Note: The directory in the file system is named 'Scenario_1_All' (with underscore)
# but the prompt requested 'Scenario_1-All'. We will use the existing directory if it exists,
# otherwise create the requested one.
save_dir = os.path.join('Models', 'Scenario_1_All')
if not os.path.exists(save_dir):
    save_dir = os.path.join('Models', 'Scenario_1-All')
    os.makedirs(save_dir, exist_ok=True)

print(f"Saving models to: {save_dir}")

for name, model in trained_models.items():
    # Create a safe filename
    filename = f"{name.replace(' ', '_').lower()}_model.pkl"
    filepath = os.path.join(save_dir, filename)
    
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)
        
    print(f"Saved {name} to {filepath}")

print("All models saved successfully!")

Saving models to: Models\Scenario_1_All
Saved Random Forest to Models\Scenario_1_All\random_forest_model.pkl
Saved SVM to Models\Scenario_1_All\svm_model.pkl
Saved XGBoost to Models\Scenario_1_All\xgboost_model.pkl
Saved KNN to Models\Scenario_1_All\knn_model.pkl
All models saved successfully!
